# Day4 - Part4: LLM을 내 목적에 맞게 길들이기, 파인튜닝(Fine-Tuning)

### 개요

Day 4의 이전 파트들에서 우리는 RAG를 통해 LLM에게 외부 지식을 참고하는 능력('오픈북')을, 에이전트를 통해 스스로 도구를 사용하는 행동력('손과 발')을 부여했습니다. 

이를 통해 LLM은 매우 유능한 정보 검색 및 작업 수행 비서가 될 수 있었습니다. 

하지만 만약 우리가 원하는 것이 단순히 정보를 찾거나 정해진 도구를 쓰는 것을 넘어, `LLM 자체의 말투, 스타일, 특정 작업에 대한 근본적인 반응 방식을 바꾸고 싶다면` 어떻게 해야 할까요?

  * 법률 문서 초안을 작성할 때, 딱딱하고 전문적인 법률 용어만 사용하도록 만들고 싶을 때
  
  * 고객 응대 챗봇이 항상 특정 브랜드의 긍정적이고 친근한 톤앤매너를 유지하게 하고 싶을 때
  * 복잡한 코드 버그 리포트를 입력하면, 정해진 JSON 형식으로 이슈를 요약하고 심각도를 평가하도록 만들고 싶을 때

이처럼 기존의 거대 언어 모델(LLM)을 특정 도메인이나 작업에 맞게 '재교육'하여, 그 행동 자체를 미세하게 조정하는 과정을 `파인튜닝(Fine-Tuning)` 또는 `미세조정`이라고 합니다. 

이는 마치 다재다능한 천재 배우(기본 LLM)에게 특정 영화의 캐릭터(우리의 목표)를 완벽하게 소화하도록 집중 연기 훈련을 시키는 것과 같습니다.

이번 파트에서는 파인튜닝이 왜 강력한 기술이며, 프롬프트 엔지니어링이나 RAG와는 어떻게 다른지 명확히 이해합니다.

그리고 OpenAI의 강력한 `gpt-4o` 모델을, 우리가 직접 준비한 데이터셋을 사용해 파인튜닝하는 전 과정을 코드를 통해 체험해 볼 것입니다. 

데이터 준비부터 파인튜닝 작업 실행, 그리고 결과 비교까지, 여러분의 손으로 직접 '맞춤형 LLM'을 만들어내는 흥미로운 여정이 될 것입니다.

`이번 파트의 학습 목표:`

  * 파인튜닝의 개념을 이해하고, 프롬프트 엔지니어링, RAG와의 차이점을 설명할 수 있습니다.
  
  * 파인튜닝이 적합한 시나리오와 부적합한 시나리오를 구별할 수 있습니다.
  * OpenAI API를 이용한 파인튜닝의 전체 프로세스(데이터 준비, 업로드, 작업 생성, 모델 사용)를 이해합니다. 
  * 파인튜닝을 위한 고품질 학습 데이터를 `JSONL 형식`으로 직접 만들고 준비할 수 있습니다. 
  * `openai` Python 라이브러리를 사용하여 파인튜닝 작업을 생성하고, 모니터링하며, 완료된 모델을 호출할 수 있습니다.
  * 기본 모델과 파인튜닝된 모델의 성능 차이를 비교 분석하여 파인튜닝의 효과를 직접 확인할 수 있습니다.
  * 최신 뉴스 기사를 '특정 관점'으로 요약하는 `맞춤형 AI 뉴스 분석가`를 파인튜닝으로 구현할 수 있습니다.



-----

### 1. 파인튜닝, 언제 사용해야 할까요? (Prompting vs. RAG vs. Fine-Tuning)

LLM을 특정 작업에 맞게 조정하는 방법은 여러 가지가 있습니다. 파인튜닝이 항상 정답은 아니며, 비용과 노력을 고려하여 가장 효율적인 방법을 선택해야 합니다. 각 기술의 특징을 비교해 봅시다.

| 구분 | `프롬프트 엔지니어링` | `RAG (검색 증강 생성)` | `파인튜닝 (Fine-Tuning)` |
| :--- | :--- | :--- | :--- |
| `핵심 아이디어` | LLM에게 보내는 `'지시사항(프롬프트)'`을 정교하게 작성하여 원하는 답변을 유도 | LLM에게 외부 지식(문서, DB)을 `'참고 자료'`로 함께 제공하여 사실 기반 답변 생성 | LLM의 `'내부 가중치(weights)'`를 새로운 데이터로 업데이트하여 모델의 행동 자체를 변화시킴 |
| `적합한 상황` | • 간단한 작업\<br\>• 스타일/포맷의 일시적 변경\<br\>• 빠른 프로토타이핑 | • 최신 정보가 필요한 경우\<br\>• 환각(Hallucination)을 줄여야 할 때\<br\>• 내부 문서 기반의 Q\&A | • 모델의 `스타일, 톤, 형식`을 일관되게 바꿔야 할 때\<br\>• 특정 도메인의 언어나 패턴을 학습시켜야 할 때\<br\>• 복잡한 지시사항을 짧은 프롬프트로 처리하고 싶을 때 |
| `장점` | • 가장 저렴하고 빠름\<br\>• 코딩 불필요 | • 지식 업데이트 용이\<br\>• 사실 기반 답변으로 신뢰도 향상 | • 더 짧은 프롬프트로 높은 성능\<br\>• API 비용 절감 가능성\<br\>• 더 일관되고 안정적인 결과 |
| `단점` | • 복잡한 작업에는 한계\<br\>• 긴 프롬프트 필요\<br\>• 일관성 부족 | • 검색 시스템 구축 필요\<br\>• 검색 품질에 성능 의존\<br\>• 모델의 근본적 행동은 불변 | • 고품질 데이터 준비 필요 \<br\>• 학습 비용 발생 \<br\>• 지식 자체를 주입하는 데는 비효율적  |

`핵심은 이것입니다:` 새로운 '지식'을 가르쳐야 한다면 `RAG`를, 모델의 '행동 방식'을 바꾸고 싶다면 `파인튜닝`을 우선적으로 고려하세요. 이 둘은 상호 보완적으로 함께 사용될 수도 있습니다.


-----

### 2. OpenAI API를 이용한 파인튜닝 프로세스

공식 문서 : [링크](https://platform.openai.com/docs/guides/fine-tuning#page-top)

과거에는 LLM 파인튜닝을 위해 강력한 GPU와 복잡한 딥러닝 프레임워크에 대한 이해가 필수적이었습니다. 

하지만 OpenAI는 API를 통해 이 과정을 매우 편리하게 만들어 제공합니다.  우리는 단지 고품질의 학습 데이터만 준비하면 됩니다.

전체 과정은 다음과 같은 4단계로 이루어집니다.

1.  `Step 1: 고품질 데이터셋 준비 (JSONL 형식)`

2.  `Step 2: 데이터 업로드 및 파인튜닝 작업 생성`
3.  `Step 3: 파인튜닝 작업 모니터링`
4.  `Step 4: 파인튜닝된 모델 사용 및 평가`

이제 각 단계를 코드로 직접 구현해 봅시다.

#### 2.1. 환경 설정 및 데이터 준비

먼저 필요한 라이브러리를 설치하고 API 키를 설정합니다.

In [1]:
!pip install openai python-dotenv -q

In [2]:
# .env 파일에서 환경 변수 로드 (반드시 OPENAI_API_KEY 환경 변수가 설정되어 있어야 합니다.)
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI 클라이언트 초기화
from openai import OpenAI
client = OpenAI()

print("OpenAI 클라이언트 준비 완료!")

OpenAI 클라이언트 준비 완료!


이제 파인튜닝의 성패를 좌우하는 가장 중요한 단계, `데이터셋 준비`입니다. OpenAI의 챗 모델(GPT-3.5, GPT-4, GPT-4o 등)을 파인튜닝하기 위해서는 데이터를 특정 `JSONL` 형식으로 만들어야 합니다. 

JSONL은 'JSON Lines'의 약자로, 파일의 한 줄(line)이 하나의 독립된 JSON 객체인 형태입니다. 각 JSON 객체는 `"messages"`라는 키를 가지며, 이 키의 값은 대화 형식의 리스트입니다. 

```json
{"messages": [{"role": "system", "content": "시스템 메시지"}, {"role": "user", "content": "사용자 질문"}, {"role": "assistant", "content": "모델의 이상적인 답변"}]}
{"messages": [{"role": "system", "content": "시스템 메시지"}, {"role": "user", "content": "두 번째 사용자 질문"}, {"role": "assistant", "content": "두 번째 이상적인 답변"}]}
```

이번 실습에서는 최신 데이터/AI 관련 뉴스 기사를 "핵심 요약"과 "주요 배경" 두 가지 항목으로 구조화하여 요약하는 모델을 만들어 보겠습니다.

In [3]:
# 1. 파인튜닝을 위한 샘플 뉴스 데이터 준비 (10개 이상)
training_data = [
    {
        "title": "데이터브릭스, 차세대 'AI 거버넌스' 프레임워크 '유니티 카탈로그 2.0' 발표",
        "content": "2025년 6월 20일, 데이터 및 AI 기업 데이터브릭스는 연례 컨퍼런스에서 차세대 데이터 거버넌스 프레임워크인 '유니티 카탈로그 2.0'을 공개했다. 이 프레임워크는 데이터, AI 모델, 관련 파이프라인 전체에 걸쳐 통합된 보안 및 거버넌스를 제공하는 것이 특징이다. 특히, 생성형 AI 모델의 출력 결과에 대한 실시간 모니터링과 유해성 탐지 기능을 탑재하여 기업이 책임감 있는 AI를 구현할 수 있도록 지원한다. 데이터브릭스의 CEO는 \"AI의 민주화는 강력한 거버넌스 위에서만 가능하다\"며, \"유니티 카탈로그 2.0은 데이터 팀과 보안 팀 사이의 간극을 메우는 핵심 솔루션이 될 것\"이라고 강조했다.",
        "source": "데이터 이코노미"
    },
    {
        "title": "AI 옵저버빌리티(Observability) 플랫폼 '뉴럴와처', 2억 달러 투자 유치",
        "content": "AI 모델의 운영 상태를 실시간으로 감시하고 분석하는 'AI 옵저버빌리티' 분야가 급성장하고 있다. 관련 스타트업 '뉴럴와처(NeuralWatcher)'는 최근 시리즈 C 펀딩에서 2억 달러 규모의 투자를 유치했다고 밝혔다. 뉴럴와처의 플랫폼은 LLM 애플리케이션에서 발생하는 데이터 드리프트, 성능 저하, 환각(Hallucination) 현상을 자동으로 탐지하고 개발자에게 경고한다. 이를 통해 기업은 AI 서비스의 신뢰도를 높이고 예상치 못한 오류로 인한 비즈니스 손실을 최소화할 수 있다. 업계 전문가들은 AI 모델이 복잡해질수록 옵저버빌리티의 중요성은 더욱 커질 것이라고 전망했다.",
        "source": "AI 스타트업 위클리"
    },
    {
        "title": "구글 클라우드, '버텍스 AI 에이전트 빌더' 정식 출시… 노코드 AI 개발 시대 열어",
        "content": "구글 클라우드는 자사의 AI 플랫폼 버텍스 AI(Vertex AI)에 '에이전트 빌더' 기능을 정식으로 추가했다고 발표했다. 이 기능은 코딩 경험이 없는 사용자도 간단한 자연어 명령어와 그래픽 인터페이스를 통해 복잡한 AI 에이전트를 구축할 수 있도록 돕는다. 사용자는 지식 베이스(문서, 웹사이트 등)를 연결하고, 구글 검색, 지도, 캘린더 등 다양한 구글 서비스를 도구로 추가하여 고객 서비스 챗봇, 사내 업무 자동화 봇 등을 손쉽게 만들 수 있다. 이는 전문 개발자뿐만 아니라 현업 실무자들도 직접 AI 솔루션을 만들 수 있는 '노코드 AI' 시대의 본격적인 시작을 의미한다.",
        "source": "클라우드 인사이트"
    },
    {
        "title": "메타, 차세대 멀티모달 AI 모델 'LLaMA-Vision 3.0' 공개",
        "content": "메타는 텍스트, 이미지, 비디오를 동시에 처리할 수 있는 멀티모달 AI 모델 'LLaMA-Vision 3.0'을 공개했다. 이 모델은 기존 LLaMA 시리즈 대비 시각적 이해 능력이 40% 향상되었으며, 실시간 비디오 분석과 3D 객체 인식 기능을 지원한다. 특히 의료 영상 분석, 자율주행차 인식 시스템, AR/VR 콘텐츠 생성 등 다양한 산업 분야에서 활용 가능하다. 메타의 AI 연구팀은 \"멀티모달 AI는 인간의 인지 능력에 한 걸음 더 가까워진 기술\"이라며, \"향후 메타버스 생태계의 핵심 기술로 자리잡을 것\"이라고 전망했다.",
        "source": "AI 테크 리뷰"
    },
    {
        "title": "오픈AI, 'GPT-5' 개발 중단 발표… 대신 'GPT-4o Ultra' 집중 개발",
        "content": "오픈AI가 차세대 언어모델 'GPT-5' 개발을 중단하고, 대신 현재 GPT-4o의 성능을 극대화한 'GPT-4o Ultra' 개발에 집중한다고 발표했다. 이는 모델 크기를 무작정 키우는 것보다 효율성과 안전성을 우선시하는 전략 변화로 해석된다. GPT-4o Ultra는 기존 모델 대비 추론 속도가 3배 빨라지고, 에너지 효율성은 50% 개선될 예정이다. 또한 코딩, 수학, 과학 분야에서의 전문성을 크게 강화하여 연구자와 개발자들의 생산성 향상에 기여할 것으로 기대된다. 업계에서는 이러한 접근법이 AI 발전의 새로운 패러다임을 제시할 것으로 평가하고 있다.",
        "source": "AI 뉴스 데일리"
    },
    {
        "title": "마이크로소프트, 'Azure AI Studio' 대폭 업데이트… 엔터프라이즈 AI 개발 환경 혁신",
        "content": "마이크로소프트가 클라우드 기반 AI 개발 플랫폼 'Azure AI Studio'의 대규모 업데이트를 발표했다. 새로운 버전은 기업용 AI 애플리케이션 개발을 위한 통합 환경을 제공하며, 데이터 전처리부터 모델 배포까지 전 과정을 하나의 플랫폼에서 처리할 수 있다. 특히 주목할 점은 '원클릭 AI 파이프라인' 기능으로, 복잡한 MLOps 과정을 자동화하여 개발 시간을 80% 단축시킨다. 또한 기업의 보안 정책에 맞춘 프라이빗 AI 모델 훈련 환경과 실시간 성능 모니터링 대시보드도 제공한다. 마이크로소프트는 \"엔터프라이즈 AI의 진입 장벽을 낮추는 것이 목표\"라고 밝혔다.",
        "source": "엔터프라이즈 IT"
    },
    {
        "title": "아마존 웹 서비스, 'AI 코파일럿' 서비스 베타 출시… 개발자 생산성 혁명",
        "content": "아마존 웹 서비스(AWS)가 개발자를 위한 AI 어시스턴트 'AWS CodeWhisperer Pro'를 정식 출시했다. 이 서비스는 자연어로 작성된 요구사항을 실제 코드로 자동 변환하고, 기존 코드의 버그를 찾아 수정 방안을 제시한다. 특히 AWS 클라우드 서비스와의 연동 코드 생성에 특화되어 있어, 클라우드 네이티브 애플리케이션 개발 시간을 대폭 단축시킨다. 베타 테스트에 참여한 개발자들은 평균 35% 이상의 개발 속도 향상을 경험했다고 보고했다. AWS는 \"AI가 개발자를 대체하는 것이 아니라, 더 창의적인 작업에 집중할 수 있도록 돕는 것\"이라고 강조했다.",
        "source": "개발자 위클리"
    },
    {
        "title": "엔비디아, 'H200 Ultra' GPU 공개… AI 훈련 성능 2배 향상",
        "content": "엔비디아가 차세대 AI 전용 GPU 'H200 Ultra'를 공개했다. 이 GPU는 기존 H100 대비 AI 모델 훈련 성능이 2배 향상되었으며, 메모리 용량도 188GB로 확대되어 더 큰 규모의 모델 훈련이 가능하다. 특히 트랜스포머 아키텍처 기반 모델의 훈련에 최적화되어 있어, LLM 개발 시간을 크게 단축시킬 수 있다. 또한 새로운 'NVLink 5.0' 기술을 통해 여러 GPU 간 데이터 전송 속도가 3배 빨라져, 대규모 분산 훈련 환경에서의 효율성이 크게 개선되었다. 엔비디아 CEO는 \"H200 Ultra는 AGI 달성을 위한 핵심 인프라\"라고 평가했다.",
        "source": "하드웨어 인사이트"
    },
    {
        "title": "스탠포드 대학, 'AI 안전성 평가' 새로운 벤치마크 'SafetyBench 2.0' 발표",
        "content": "스탠포드 대학 AI 연구소가 AI 모델의 안전성을 종합적으로 평가하는 새로운 벤치마크 'SafetyBench 2.0'을 발표했다. 이 벤치마크는 편향성, 독성, 환각, 개인정보 유출 등 12개 카테고리에서 AI 모델의 위험성을 정량적으로 측정한다. 기존 평가 방식과 달리 실제 사용 환경을 모방한 시나리오 기반 테스트를 도입하여 더욱 현실적인 평가가 가능하다. 연구팀은 주요 AI 모델들을 대상으로 한 평가 결과, 대부분의 모델이 특정 영역에서 예상보다 높은 위험성을 보였다고 발표했다. 이는 AI 안전성 연구의 중요성을 다시 한번 부각시키는 결과로 평가된다.",
        "source": "AI 안전성 저널"
    },
    {
        "title": "테슬라, 'FSD(Full Self-Driving) v13' 업데이트… 도심 주행 성공률 95% 달성",
        "content": "테슬라가 완전 자율주행 소프트웨어 'FSD v13' 업데이트를 배포했다. 이번 버전은 복잡한 도심 환경에서의 주행 성공률이 95%에 달해, 상용화 수준에 근접했다는 평가를 받고 있다. 특히 교차로 좌회전, 보행자 회피, 공사 구간 우회 등 까다로운 상황에서의 판단 능력이 크게 향상되었다. 이는 테슬라가 독자 개발한 '뉴럴 네트워크 v4.0'과 실시간 경로 최적화 알고리즘의 결과물이다. 테슬라는 \"2025년 말까지 완전 무인 택시 서비스를 시작할 계획\"이라며, \"FSD 기술이 교통 혁명의 출발점이 될 것\"이라고 전망했다.",
        "source": "자율주행 테크"
    },
    {
        "title": "IBM, 양자 컴퓨팅과 AI 융합 기술 'Quantum-AI Hybrid' 플랫폼 공개",
        "content": "IBM이 양자 컴퓨팅과 인공지능을 결합한 혁신적인 플랫폼 'Quantum-AI Hybrid'를 공개했다. 이 플랫폼은 양자 컴퓨터의 병렬 처리 능력과 AI의 패턴 인식 능력을 결합하여, 기존 컴퓨터로는 해결하기 어려운 복잡한 최적화 문제를 해결한다. 특히 신약 개발, 금융 리스크 분석, 물류 최적화 등의 분야에서 기존 방식 대비 1000배 빠른 처리 속도를 보여준다. IBM 연구진은 \"양자-AI 융합 기술은 컴퓨팅 패러다임의 근본적 변화를 가져올 것\"이라며, \"향후 10년 내 상용화를 목표로 하고 있다\"고 밝혔다.",
        "source": "퀀텀 컴퓨팅 리뷰"
    },
    {
        "title": "바이두, 중국어 특화 AI 모델 'ERNIE 5.0' 출시… GPT-4 수준 성능 달성",
        "content": "중국 IT 기업 바이두가 중국어 처리에 특화된 대규모 언어모델 'ERNIE 5.0'을 출시했다. 이 모델은 중국어 이해와 생성 능력에서 GPT-4와 동등한 수준의 성능을 보여주며, 특히 중국 문화와 역사에 관련된 질문에서는 더 우수한 답변을 제공한다. ERNIE 5.0은 14억 개의 중국어 웹 페이지와 5000만 권의 중국어 도서를 학습 데이터로 활용했으며, 중국어의 미묘한 뉘앙스와 문맥을 정확히 파악할 수 있다. 바이두는 \"글로벌 AI 경쟁에서 중국어권 사용자들에게 최적화된 서비스를 제공하는 것이 목표\"라고 밝혔다.",
        "source": "차이나 AI 리포트"
    }
]

In [4]:
# 2. 데이터를 OpenAI 파인튜닝 형식(JSONL)으로 변환
import json

# 시스템 메시지는 모델의 역할을 정의합니다.
system_message = "당신은 IT 전문 기자입니다. 주어진 뉴스 기사를 '핵심 요약'과 '주요 배경'으로 나누어 구조화된 형식으로 요약해야 합니다."
jsonl_data = []

for article in training_data:
    user_prompt = f"다음 뉴스 기사를 요약해 주세요:\n\n제목: {article['title']}\n내용: {article['content']}"

    # 각 기사에 대한 '이상적인 답변'을 우리가 직접 만들어 제공합니다.
    # 이것이 모델이 학습할 목표(Target)가 됩니다.
    if "메타" in article['title']:
        ideal_response = """
- 핵심 요약: 메타가 텍스트, 이미지, 비디오를 동시에 처리할 수 있는 멀티모달 AI 모델 'LLaMA-Vision 3.0'을 공개했습니다.
- 주요 배경: 기존 LLaMA 시리즈 대비 시각적 이해 능력이 40% 향상되었으며, 의료 영상 분석, 자율주행차 인식 시스템, AR/VR 콘텐츠 생성 등 다양한 산업 분야에서 활용 가능한 멀티모달 AI 기술의 발전이 필요했기 때문입니다.
"""
    elif "오픈AI" in article['title']:
        ideal_response = """
- 핵심 요약: 오픈AI가 차세대 언어모델 'GPT-5' 개발을 중단하고, 대신 'GPT-4o Ultra' 개발에 집중한다고 발표했습니다.
- 주요 배경: 모델 크기를 무작정 키우는 것보다 효율성과 안전성을 우선시하는 전략 변화로, 추론 속도 3배 향상과 에너지 효율성 50% 개선을 통해 AI 발전의 새로운 패러다임을 제시하기 위함입니다.
"""
    elif "마이크로소프트" in article['title']:
        ideal_response = """
- 핵심 요약: 마이크로소프트가 클라우드 기반 AI 개발 플랫폼 'Azure AI Studio'의 대규모 업데이트를 발표했습니다.
- 주요 배경: 기업용 AI 애플리케이션 개발을 위한 통합 환경 제공과 '원클릭 AI 파이프라인' 기능을 통해 복잡한 MLOps 과정을 자동화하여 개발 시간을 80% 단축시키고 엔터프라이즈 AI의 진입 장벽을 낮추기 위함입니다.
"""
    elif "아마존" in article['title']:
        ideal_response = """
- 핵심 요약: 아마존 웹 서비스(AWS)가 개발자를 위한 AI 어시스턴트 'AWS CodeWhisperer Pro'를 정식 출시했습니다.
- 주요 배경: 자연어로 작성된 요구사항을 실제 코드로 자동 변환하고 기존 코드의 버그를 찾아 수정하는 기능을 통해 개발자들이 더 창의적인 작업에 집중할 수 있도록 돕기 위함입니다.
"""
    elif "엔비디아" in article['title']:
        ideal_response = """
- 핵심 요약: 엔비디아가 차세대 AI 전용 GPU 'H200 Ultra'를 공개했습니다.
- 주요 배경: 기존 H100 대비 AI 모델 훈련 성능이 2배 향상되고 메모리 용량이 188GB로 확대되어 더 큰 규모의 모델 훈련이 가능하며, AGI 달성을 위한 핵심 인프라 구축이 필요했기 때문입니다.
"""
    elif "스탠포드" in article['title']:
        ideal_response = """
- 핵심 요약: 스탠포드 대학 AI 연구소가 AI 모델의 안전성을 종합적으로 평가하는 새로운 벤치마크 'SafetyBench 2.0'을 발표했습니다.
- 주요 배경: 편향성, 독성, 환각, 개인정보 유출 등 12개 카테고리에서 AI 모델의 위험성을 정량적으로 측정하고, 실제 사용 환경을 모방한 시나리오 기반 테스트를 통해 AI 안전성 연구의 중요성을 부각시키기 위함입니다.
"""
    elif "테슬라" in article['title']:
        ideal_response = """
- 핵심 요약: 테슬라가 완전 자율주행 소프트웨어 'FSD v13' 업데이트를 배포하여 도심 주행 성공률 95%를 달성했습니다.
- 주요 배경: 복잡한 도심 환경에서의 교차로 좌회전, 보행자 회피, 공사 구간 우회 등 까다로운 상황에서의 판단 능력 향상을 통해 2025년 말 완전 무인 택시 서비스 시작을 목표로 하는 교통 혁명을 준비하기 위함입니다.
"""
    elif "IBM" in article['title']:
        ideal_response = """
- 핵심 요약: IBM이 양자 컴퓨팅과 인공지능을 결합한 혁신적인 플랫폼 'Quantum-AI Hybrid'를 공개했습니다.
- 주요 배경: 양자 컴퓨터의 병렬 처리 능력과 AI의 패턴 인식 능력을 결합하여 신약 개발, 금융 리스크 분석, 물류 최적화 등의 분야에서 기존 방식 대비 1000배 빠른 처리 속도를 실현하고 컴퓨팅 패러다임의 근본적 변화를 가져오기 위함입니다.
"""
    else: # 바이두
        ideal_response = """
- 핵심 요약: 중국 IT 기업 바이두가 중국어 처리에 특화된 대규모 언어모델 'ERNIE 5.0'을 출시했습니다.
- 주요 배경: 중국어 이해와 생성 능력에서 GPT-4와 동등한 수준의 성능을 보여주며, 14억 개의 중국어 웹 페이지와 5000만 권의 중국어 도서를 학습하여 글로벌 AI 경쟁에서 중국어권 사용자들에게 최적화된 서비스를 제공하기 위함입니다.
"""

    jsonl_data.append({
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": ideal_response.strip()}
        ]
    })

# JSONL 파일로 저장
os.makedirs("../datasets/openai", exist_ok=True)
file_name = "../datasets/openai/news_tuning_data.jsonl"
with open(file_name, "w", encoding="utf-8") as f:
    for entry in jsonl_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"'{file_name}' 파일이 성공적으로 생성되었습니다.")
print(f"총 {len(jsonl_data)}개의 학습 데이터가 포함되어 있습니다.")

'../datasets/openai/news_tuning_data.jsonl' 파일이 성공적으로 생성되었습니다.
총 12개의 학습 데이터가 포함되어 있습니다.


#### 2.2. 데이터 업로드 및 파인튜닝 작업 생성

이제 준비된 `news_tuning_data.jsonl` 파일을 OpenAI 서버에 업로드하고, 이 파일을 사용하여 `gpt-4o` 모델에 대한 파인튜닝 작업을 시작합니다.

In [5]:
# 1. 파일 업로드
training_file = client.files.create(
  file=open(file_name, "rb"),
  purpose="fine-tune"
)

# gpt-4.1-mini-2025-04-14 모델을 기반으로 파인튜닝을 시작합니다.print(f"파일이 성공적으로 업로드되었습니다. 파일 ID: {training_file.id}")

In [6]:
# gpt-4.1-mini-2025-04-14 모델을 기반으로 파인튜닝을 시작합니다.
target_model = "gpt-4.1-mini-2025-04-14"

In [7]:
# 2. 파인튜닝 작업 생성
fine_tuning_job = client.fine_tuning.jobs.create(
  training_file=training_file.id,
  model="gpt-4.1-mini-2025-04-14"
)

# 생성된 작업의 ID와 상태를 저장해 둡니다.
job_id = fine_tuning_job.id
status = fine_tuning_job.status

print(f"파인튜닝 작업이 시작되었습니다. 작업 ID: {job_id}, 상태: {status}")

파인튜닝 작업이 시작되었습니다. 작업 ID: ftjob-JBVvxZr76BFNxDLZkZrFxlmM, 상태: validating_files


`참고:` 파인튜닝에는 비용이 발생합니다.  비용은 학습 데이터의 토큰 수와 학습 반복 횟수(epoch)에 따라 결정됩니다.  

작은 규모의 테스트라도 비용이 청구되므로, OpenAI의 최신 요금 정책을 반드시 확인하세요.


#### 2.3. 파인튜닝 작업 모니터링

파인튜닝 작업은 데이터셋 크기에 따라 수 분에서 수 시간까지 걸릴 수 있습니다. `retrieve` API를 사용하여 작업 상태를 주기적으로 확인할 수 있습니다.

In [ ]:
# openai 클라우드 서버에서 finetunning을 진행해줌

In [8]:
import time

# 작업이 완료될 때까지 1분마다 상태 확인
while status not in ["succeeded", "failed"]:
    time.sleep(60)
    job_info = client.fine_tuning.jobs.retrieve(job_id)
    status = job_info.status
    print(f"진행 상황 확인 중... 상태: {status}")

print("\n--- 작업 상세 정보 ---")
print(job_info)
print("--------------------")

# 작업이 성공적으로 완료되면, 파인튜닝된 모델의 ID를 저장합니다.
if status == 'succeeded':
    fine_tuned_model_id = job_info.fine_tuned_model
    print(f"파인튜닝이 성공적으로 완료되었습니다!")
    print(f"새로운 모델 ID: {fine_tuned_model_id}")
else:
    print(f"파인튜닝 작업이 실패했습니다. 상태: {status}")

진행 상황 확인 중... 상태: validating_files
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: running
진행 상황 확인 중... 상태: succeeded

--- 작업 상세 정보 ---
FineTuningJob(id='ftjob-JBVvxZr76BFNxDLZkZrFxlmM', created_at=1750925647, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-mini-2025-04-14:personal::BmcC5yzt', finished_at=1750926258, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=8), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-iTYxohqDzbuhBEPa8UrBfX6b', result_files=['

#### 2.4. 파인튜닝된 모델 사용 및 평가

이제 대망의 순간입니다\! 우리가 만든 맞춤형 모델과 기존 `gpt-4o` 모델에 동일한 요청을 보내고, 그 결과를 비교하여 파인튜닝의 효과를 직접 확인해 봅시다.

새로운 뉴스 기사를 준비하여 테스트합니다.

In [9]:
# 테스트용 새로운 뉴스 기사
new_article = {
    "title": "메타, 차세대 언어 모델 '라마 4' 공개…온디바이스 AI 시장 정조준",
    "content": "페이스북의 모회사 메타가 최신 거대 언어 모델(LLM)인 '라마 4(Llama 4)'를 공개했습니다. 라마 4는 이전 모델보다 크기는 줄이면서도 성능은 대폭 향상시킨 것이 특징입니다. 특히, 스마트폰과 같은 개인 기기에서 직접 구동되는 온디바이스 AI 환경에 최적화되어, 인터넷 연결 없이도 빠른 속도로 고품질의 AI 기능을 제공할 수 있습니다. 메타는 라마 4를 오픈소스로 공개하여, 더 많은 개발자들이 AI 기술에 접근하고 혁신을 가속화할 수 있도록 지원할 것이라고 밝혔습니다. 이는 AI 시장의 주도권을 잡기 위한 메타의 중요한 전략으로 평가됩니다."
}
test_prompt = f"다음 뉴스 기사를 요약해 주세요:\n\n제목: {new_article['title']}\n내용: {new_article['content']}"

# 비교를 위한 기본 gpt-4.1-mini 모델 호출
print("--- 1. 기본 gpt-4.1-mini 모델 응답 ---")
base_model_response = client.chat.completions.create(
    model=target_model,
    messages=[
        {"role": "system", "content": system_message}, # 파인튜닝 때와 동일한 시스템 메시지
        {"role": "user", "content": test_prompt}
    ]
)
print(base_model_response.choices[0].message.content)

--- 1. 기본 gpt-4.1-mini 모델 응답 ---
핵심 요약:
메타가 최신 거대 언어 모델 '라마 4(Llama 4)'를 공개했다. 라마 4는 크기를 줄이면서도 성능을 크게 향상시켰으며, 개인 스마트폰 등 온디바이스 AI 환경에 최적화돼 인터넷 연결 없이도 빠르고 고품질 AI 기능을 제공한다. 이 모델은 오픈소스로 공개되어 개발자들의 접근성과 혁신을 지원한다.

주요 배경:
메타는 AI 시장에서의 경쟁력을 강화하고 주도권을 확보하기 위해 라마 4를 발표했다. 온디바이스 AI는 개인정보 보호 및 신속한 처리 요구가 높아지는 가운데 중요한 시장으로 부상하고 있으며, 메타는 이를 전략적으로 겨냥해 차별화된 AI 서비스를 제공하려는 의도를 가지고 있다. 오픈소스화는 AI 기술 발전을 빠르게 촉진하고 생태계 확장에 기여할 전망이다.


In [10]:
# 파인튜닝된 우리 모델 호출 (fine_tuned_model_id 변수에 저장된 ID 사용)
print("--- 2. 파인튜닝된 모델 응답 ---")
if 'fine_tuned_model_id' in locals() and fine_tuned_model_id:
    fine_tuned_model_response = client.chat.completions.create(
        model=fine_tuned_model_id,
        messages=[
            # 파인튜닝된 모델은 이미 역할을 학습했으므로, 시스템 메시지를 생략하거나 간단하게 할 수 있습니다.
            {"role": "user", "content": test_prompt}
        ]
    )
    print(fine_tuned_model_response.choices[0].message.content)
else:
    print("파인튜닝된 모델 ID를 사용할 수 없습니다.")

--- 2. 파인튜닝된 모델 응답 ---
메타가 최신 거대 언어 모델 '라마 4(Llama 4)'를 공개했습니다. 이전 모델보다 크기는 줄이면서도 성능은 대폭 향상시킨 것이 특징이며, 스마트폰과 같은 개인 기기에서 직접 구동되는 온디바이스 AI 환경에 최적화되어 인터넷 연결 없이도 빠른 속도로 고품질의 AI 기능을 제공할 수 있습니다. 메타는 라마 4를 오픈소스로 공개하여 더 많은 개발자들이 AI 기술에 접근하고 혁신을 가속화할 수 있도록 지원하며, 이는 AI 시장의 주도권을 잡기 위한 메타의 중요한 전략으로 평가됩니다.


> 모델이 접근이 안되어 에러가 나시는 분들은 https://platform.openai.com/settings/ 로 접속하셔서 
> 
> Limits 라는 메뉴에서 해당 프로젝트에서 접근 가능한 모델로 선택되어 있는지 확인하세요!